In [ ]:
"""
╔══════════════════════════════════════════════════════════════════════════╗
║   NIFTY Greek-Based Options Backtester  — 100% FREE, No API Key         ║
║   Strategy : Iron Condor (safer than Short Straddle)                    ║
║              Sell OTM Call + Sell OTM Put                               ║
║              Buy further OTM Call + Buy further OTM Put (wings)         ║
║   Data     : yfinance (spot) + Black-Scholes IV simulation              ║
║   Based on : IIT-K Options Greeks PS Framework                          ║
╠══════════════════════════════════════════════════════════════════════════╣
║  HOW TO RUN IN GOOGLE COLAB:                                            ║
║  Cell 1 →  !pip install yfinance pandas numpy scipy plotly -q           ║
║  Cell 2 →  exec(open('greek_backtester_free.py').read())                ║
║            run_backtest()                                               ║
╚══════════════════════════════════════════════════════════════════════════╝

  IRON CONDOR STRUCTURE (per trade):
  ─────────────────────────────────────────────────────────────────────
  SELL  Call at K_call  (ATM + CALL_OFFSET strikes above spot)
  BUY   Call at K_call_wing  (K_call + WING_WIDTH strikes above)
  SELL  Put  at K_put   (ATM - PUT_OFFSET  strikes below spot)
  BUY   Put  at K_put_wing   (K_put  - WING_WIDTH strikes below)
  ─────────────────────────────────────────────────────────────────────
  Net premium = (sell_call - buy_call) + (sell_put - buy_put)
  Max loss    = (WING_WIDTH × STRIKE_ROUND) - net_premium  [per unit]
  ─────────────────────────────────────────────────────────────────────
"""

# ── INSTALL (uncomment if running in Colab) ───────────────────────────────
# !pip install yfinance pandas numpy scipy plotly -q

import yfinance as yf
import pandas as pd
import numpy as np
from scipy.stats import norm
from scipy.optimize import brentq
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings, json, datetime
warnings.filterwarnings('ignore')

pd.set_option('display.float_format', '{:.2f}'.format)

# ═══════════════════════════════════════════════════════════════════════════
# SECTION 1 — CONFIGURATION  (edit these if needed)
# ═══════════════════════════════════════════════════════════════════════════

CFG = {
    # ── Backtest window ──────────────────────────────────────
    'FROM_DATE'          : '2022-01-01',
    'TO_DATE'            : '2024-12-31',

    # ── Capital ──────────────────────────────────────────────
    'INITIAL_CAPITAL'    : 500_000,     # ₹5 lakh
    'LOT_SIZE'           : 50,          # NIFTY lot size (as per PS)
    'MAX_LOTS'           : 3,           # conservative — 3 lots max
    'CAPITAL_RISK_PCT'   : 0.12,        # 12% of cash per trade

    # ── Options model ────────────────────────────────────────
    'RISK_FREE_RATE'     : 0.065,       # 6.5% p.a.
    'IV_PREMIUM'         : 0.03,        # IV = HV20 + 3%

    # ── Regime filter (use rolling IV vs HV ratio) ────────────
    # Iron Condor: enter only when IV is elevated (sell expensive vol)
    'IC_IV_HV_MIN'       : 1.10,        # IC: IV must be > 110% of HV
    # Long Straddle: enter only when IV is depressed (buy cheap vol)
    'LS_IV_HV_MAX'       : 1.00,        # LS: IV must be < 100% of HV (at or below HV)

    # ── Entry timing ─────────────────────────────────────────
    'ENTRY_DTE'          : 30,          # 30 DTE — more time for big move to occur
    'STRIKE_ROUND'       : 50,

    # ── Iron Condor strikes ───────────────────────────────────
    # Wider OTM = more room for spot to move = safer but less premium
    'CALL_OFFSET'        : 5,           # 5×50 = 250pts above ATM  (~1SD)
    'PUT_OFFSET'         : 5,           # 5×50 = 250pts below ATM  (~1SD)
    'WING_WIDTH'         : 3,           # 3×50 = 150pts protection beyond short

    # ── Greek thresholds ─────────────────────────────────────
    'DELTA_ALERT'        : 0.20,
    'DELTA_HARD_EXIT'    : 0.35,
    'GAMMA_ALERT'        : 0.0012,
    'GAMMA_EXIT'         : 0.0018,
    'VEGA_IV_ALERT_PCT'  : 0.20,
    'VEGA_IV_EXIT_PCT'   : 0.40,

    # ── Iron Condor exits ─────────────────────────────────────
    'IC_PROFIT_TARGET'   : 0.50,        # exit at 50% of net premium collected
    'IC_STOP_LOSS_MULT'  : 2.0,         # exit if cost to close = 2× premium received
    'IC_MIN_DTE'         : 4,

    # ── Long Straddle exits ───────────────────────────────────
    'LS_PROFIT_TARGET'   : 0.50,        # hold until 50% gain — let winners run
    'LS_STOP_LOSS_PCT'   : 0.25,        # cut fast at 25% loss — theta kills slow losers
    'LS_IV_CRUSH_EXIT'   : 0.12,        # exit if IV drops 12% (tighter IV crush guard)
    'LS_MIN_DTE'         : 7,
    'LS_MAX_HOLD_DAYS'   : 15,          # 15 days max — 30 DTE entry gives more runway
}

# ═══════════════════════════════════════════════════════════════════════════
# SECTION 2 — BLACK-SCHOLES ENGINE
# ═══════════════════════════════════════════════════════════════════════════

def _bs_price(S, K, T, r, sigma, opt):
    """Raw B-S option price."""
    if T <= 0 or sigma <= 0:
        return max(S - K, 0) if opt == 'CE' else max(K - S, 0)
    d1 = (np.log(S / K) + (r + 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    if opt == 'CE':
        return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    else:
        return K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)


def greeks(S, K, T, r, sigma, opt):
    """
    Returns dict with: price, delta, gamma, vega, theta, iv
    vega  = per 1% change in IV
    theta = per calendar day
    """
    if T <= 1e-6 or sigma <= 1e-6:
        return dict(price=0, delta=0, gamma=0, vega=0, theta=0, iv=sigma)

    d1 = (np.log(S / K) + (r + 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)

    price  = _bs_price(S, K, T, r, sigma, opt)
    delta  = norm.cdf(d1) if opt == 'CE' else norm.cdf(d1) - 1
    gamma  = norm.pdf(d1) / (S * sigma * np.sqrt(T))
    vega   = S * norm.pdf(d1) * np.sqrt(T) / 100
    th_raw = -(S * norm.pdf(d1) * sigma) / (2 * np.sqrt(T))

    if opt == 'CE':
        theta = (th_raw - r * K * np.exp(-r * T) * norm.cdf(d2))  / 365
    else:
        theta = (th_raw + r * K * np.exp(-r * T) * norm.cdf(-d2)) / 365

    return dict(price=max(price, 0), delta=delta, gamma=gamma,
                vega=vega, theta=theta, iv=sigma)


# ═══════════════════════════════════════════════════════════════════════════
# SECTION 3 — DATA LAYER
# ═══════════════════════════════════════════════════════════════════════════

def fetch_nifty(from_date: str, to_date: str) -> pd.DataFrame:
    print(f"📥  Downloading NIFTY 50  [{from_date} → {to_date}] ...")
    df = yf.download("^NSEI", start=from_date, end=to_date,
                     auto_adjust=True, progress=False)
    if df.empty:
        raise RuntimeError("yfinance returned empty data. Check dates / internet.")
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    df = df[['Open', 'High', 'Low', 'Close', 'Volume']].copy()
    df.index = pd.to_datetime(df.index).tz_localize(None)
    df.dropna(inplace=True)

    log_ret = np.log(df['Close'] / df['Close'].shift(1))

    # 10-day HV (short-term noise)
    df['HV10'] = log_ret.rolling(10).std() * np.sqrt(252)
    # 20-day HV (medium-term baseline)
    df['HV20'] = log_ret.rolling(20).std() * np.sqrt(252)
    # 30-day HV (slow baseline for regime)
    df['HV30'] = log_ret.rolling(30).std() * np.sqrt(252)

    df['HV10']  = df['HV10'].bfill().fillna(0.15)
    df['HV20']  = df['HV20'].bfill().fillna(0.15)
    df['HV30']  = df['HV30'].bfill().fillna(0.15)

    # IV estimate: HV20 + small risk premium
    # Use HV30 as the "fair" baseline — HV20 as the near-term signal
    df['IV']    = (df['HV20'] + CFG['IV_PREMIUM']).clip(0.10, 0.80)

    # IV/HV ratio — key regime signal
    # > 1.10  → IV elevated → good for selling vol (Iron Condor)
    # < 0.95  → IV depressed → good for buying vol (Long Straddle)
    df['IV_HV_ratio'] = df['IV'] / df['HV30']

    # Realized vs implied gap — extra signal
    df['vol_gap'] = df['IV'] - df['HV10']   # positive = selling edge

    print(f"    ✅  {len(df)} trading days loaded")
    print(f"    IV/HV ratio range: "
          f"{df['IV_HV_ratio'].min():.2f} – {df['IV_HV_ratio'].max():.2f}  "
          f"| mean: {df['IV_HV_ratio'].mean():.2f}")
    return df


def get_monthly_expiry_dates(trading_dates: pd.DatetimeIndex,
                              from_date: str, to_date: str) -> list:
    """
    Generate last-Thursday-of-each-month expiry dates that fall on trading days.
    If a Thursday is a holiday, use the previous Wednesday, etc.
    """
    fd = pd.Timestamp(from_date)
    td = pd.Timestamp(to_date)
    expiries = []

    current = fd.to_period('M')
    end_per = td.to_period('M')

    while current <= end_per:
        year  = current.year
        month = current.month
        # last day of month
        last_day = pd.Timestamp(year=year, month=month,
                                day=pd.Timestamp(year, month, 1).days_in_month)
        # walk back to Thursday (weekday=3)
        while last_day.weekday() != 3:
            last_day -= pd.Timedelta(days=1)

        # If that Thursday is a market holiday, step back to nearest trading day
        cand = last_day
        for _ in range(5):
            if cand in trading_dates:
                expiries.append(cand)
                break
            cand -= pd.Timedelta(days=1)

        current += 1

    return sorted(set(expiries))


# ═══════════════════════════════════════════════════════════════════════════
# SECTION 4 — BACKTESTING ENGINE
# ═══════════════════════════════════════════════════════════════════════════

class IronCondorBacktester:
    """
    Iron Condor backtester — safer than short straddle due to defined max loss.

    Structure per trade:
      SELL Call at ATM + CALL_OFFSET  (short call strike)
      BUY  Call at ATM + CALL_OFFSET + WING_WIDTH  (long call wing)
      SELL Put  at ATM - PUT_OFFSET   (short put strike)
      BUY  Put  at ATM - PUT_OFFSET  - WING_WIDTH  (long put wing)

    Net premium received = (short_call - long_call) + (short_put - long_put)
    Max loss per lot     = WING_WIDTH × STRIKE_ROUND - net_premium
    Max profit per lot   = net_premium (if spot stays between short strikes at expiry)

    Greek management rules (from IIT-K PS):
      • Delta alert/exit if net |Δ| > 0.20 / 0.30
      • Vega alert/exit if IV rises > 20% / 35% above entry IV
      • Gamma alert/exit if |Γ| > 0.0015 / 0.002
      • Profit target : 30% of net premium collected
      • Stop loss     : net debit doubles (2× net premium)
      • Time exit     : ≤ 5 DTE
    """

    def __init__(self, df: pd.DataFrame):
        self.df     = df
        self.r      = CFG['RISK_FREE_RATE']
        self.cash   = CFG['INITIAL_CAPITAL']
        self.trades = []
        self.alerts = []
        self.pv_log = []
        self._tid   = 0

    # ── PUBLIC ────────────────────────────────────────────────────────────
    def run(self, from_date: str, to_date: str):
        expiries = get_monthly_expiry_dates(self.df.index, from_date, to_date)
        print(f"\n📅  {len(expiries)} monthly expiry cycles to simulate\n")
        for exp in expiries:
            self._run_cycle(exp)
        self._flush_pv_log()
        print(f"\n✅  Backtest complete — {len(self.trades)} trades")
        return self.trades, self.alerts, self.pv_log

    # ── PRIVATE — one expiry cycle ────────────────────────────────────────
    def _run_cycle(self, expiry: pd.Timestamp):
        all_dates  = self.df.index
        pre_expiry = all_dates[all_dates < expiry]

        if len(pre_expiry) < CFG['ENTRY_DTE'] + 5:
            return

        entry_date = pre_expiry[-CFG['ENTRY_DTE']]
        entry_row  = self.df.loc[entry_date]
        S0         = float(entry_row['Close'])
        iv0        = float(entry_row['IV'])
        hv0        = float(entry_row['HV20'])

        # Entry filter: IV must be elevated for selling vol
        iv_hv_ratio = float(entry_row.get('IV_HV_ratio', iv0 / hv0))
        if iv_hv_ratio < CFG['IC_IV_HV_MIN']:
            print(f"  ⏭   {expiry.date()} — IV/HV={iv_hv_ratio:.2f} < {CFG['IC_IV_HV_MIN']} (vol not elevated), skip")
            return

        step = CFG['STRIKE_ROUND']
        ATM  = round(S0 / step) * step

        # ── Four strikes of the Iron Condor ───────────────────────────────
        K_sc = ATM + CFG['CALL_OFFSET'] * step          # short call
        K_lc = K_sc + CFG['WING_WIDTH'] * step          # long  call (wing)
        K_sp = ATM - CFG['PUT_OFFSET']  * step          # short put
        K_lp = K_sp - CFG['WING_WIDTH'] * step          # long  put  (wing)

        T0   = CFG['ENTRY_DTE'] / 365

        # Entry prices & Greeks for all 4 legs
        sc0  = greeks(S0, K_sc, T0, self.r, iv0, 'CE')   # sell
        lc0  = greeks(S0, K_lc, T0, self.r, iv0, 'CE')   # buy
        sp0  = greeks(S0, K_sp, T0, self.r, iv0, 'PE')   # sell
        lp0  = greeks(S0, K_lp, T0, self.r, iv0, 'PE')   # buy

        # Net premium received (credit spread)
        call_spread_credit = sc0['price'] - lc0['price']
        put_spread_credit  = sp0['price'] - lp0['price']
        net_prem0          = call_spread_credit + put_spread_credit

        if net_prem0 <= 0:
            print(f"  ⏭   {expiry.date()} — zero/negative net premium, skipping")
            return

        # Max loss per lot (defined risk — key advantage over short straddle)
        max_loss_per_lot = (CFG['WING_WIDTH'] * step - net_prem0) * CFG['LOT_SIZE']

        # Net Greeks at entry (short legs dominate; long legs partially offset)
        net_d0 = -(sc0['delta'] - lc0['delta']) - (sp0['delta'] - lp0['delta'])
        net_g0 = -(sc0['gamma'] - lc0['gamma']) - (sp0['gamma'] - lp0['gamma'])
        net_v0 = -(sc0['vega']  - lc0['vega'])  - (sp0['vega']  - lp0['vega'])
        net_t0 = -(sc0['theta'] - lc0['theta']) - (sp0['theta'] - lp0['theta'])

        # Lot sizing: size by premium collected, but hard-cap by max loss risk
        risk_by_premium  = self.cash * CFG['CAPITAL_RISK_PCT']
        lots_by_premium  = max(1, int(risk_by_premium / (net_prem0 * CFG['LOT_SIZE'])))
        lots_by_maxloss  = max(1, int(risk_by_premium / max_loss_per_lot)) if max_loss_per_lot > 0 else 1
        lots = min(lots_by_premium, lots_by_maxloss, CFG['MAX_LOTS'])

        self._tid += 1
        trade = {
            'id'                  : self._tid,
            'strategy'            : 'Iron Condor',
            'expiry'              : str(expiry.date()),
            'entry_date'          : str(entry_date.date()),
            'entry_spot'          : round(S0, 2),
            'ATM'                 : ATM,
            'K_short_call'        : K_sc,
            'K_long_call'         : K_lc,
            'K_short_put'         : K_sp,
            'K_long_put'          : K_lp,
            'entry_iv'            : round(iv0, 4),
            'entry_hv'            : round(hv0, 4),
            'entry_sc_px'         : round(sc0['price'], 2),
            'entry_lc_px'         : round(lc0['price'], 2),
            'entry_sp_px'         : round(sp0['price'], 2),
            'entry_lp_px'         : round(lp0['price'], 2),
            'call_spread_credit'  : round(call_spread_credit, 2),
            'put_spread_credit'   : round(put_spread_credit, 2),
            'entry_net_premium'   : round(net_prem0, 2),
            'max_loss_per_lot'    : round(max_loss_per_lot, 2),
            'entry_delta'         : round(net_d0, 4),
            'entry_gamma'         : round(net_g0, 5),
            'entry_vega'          : round(net_v0, 4),
            'entry_theta'         : round(net_t0, 4),
            'lots'                : lots,
            # exit fields
            'exit_date'           : None,
            'exit_spot'           : None,
            'exit_net_premium'    : None,
            'exit_reason'         : None,
            'exit_delta'          : None,
            'exit_iv'             : None,
            'pnl'                 : None,
            'dte_held'            : None,
        }

        print(f"  ▶  Entry {entry_date.date()} | Expiry {expiry.date()} | "
              f"Spot ₹{S0:.0f} | "
              f"Condor [{K_sp}/{K_sc}–{K_lc}/{K_lp}] | "
              f"Net Prem ₹{net_prem0:.1f} | "
              f"MaxLoss ₹{max_loss_per_lot:.0f} | Lots {lots}")

        # ── Daily monitoring ─────────────────────────────────────────────
        cycle_dates = all_dates[(all_dates >= entry_date) & (all_dates <= expiry)]
        exited      = False

        for day_i, date in enumerate(cycle_dates):
            row     = self.df.loc[date]
            S       = float(row['Close'])
            iv_now  = float(row['IV'])
            dte_rem = len(cycle_dates) - day_i

            T = max(dte_rem / 365, 1e-6)

            # Current prices for all 4 legs
            sc_now = greeks(S, K_sc, T, self.r, iv_now, 'CE')
            lc_now = greeks(S, K_lc, T, self.r, iv_now, 'CE')
            sp_now = greeks(S, K_sp, T, self.r, iv_now, 'PE')
            lp_now = greeks(S, K_lp, T, self.r, iv_now, 'PE')

            # Current net premium to close (buy back shorts, sell long wings)
            curr_call_spread = sc_now['price'] - lc_now['price']
            curr_put_spread  = sp_now['price'] - lp_now['price']
            curr_net_prem    = curr_call_spread + curr_put_spread

            # P&L = net premium received at entry - net cost to close now
            pnl = (net_prem0 - curr_net_prem) * lots * CFG['LOT_SIZE']

            # Net portfolio Greeks (short legs - long legs, all negated)
            net_delta = -((sc_now['delta'] - lc_now['delta']) +
                          (sp_now['delta'] - lp_now['delta']))
            net_gamma = -((sc_now['gamma'] - lc_now['gamma']) +
                          (sp_now['gamma'] - lp_now['gamma']))
            net_vega  = -((sc_now['vega']  - lc_now['vega'])  +
                          (sp_now['vega']  - lp_now['vega']))
            net_theta = -((sc_now['theta'] - lc_now['theta']) +
                          (sp_now['theta'] - lp_now['theta']))

            iv_chg_pct = (iv_now - iv0) / iv0
            date_str   = str(date.date())

            # ── Portfolio value log ──────────────────────────────────────
            self.pv_log.append({
                'date'      : date_str,
                'value'     : round(self.cash + pnl, 2),
                'open_pnl'  : round(pnl, 2),
                'net_delta' : round(net_delta, 4),
                'net_gamma' : round(net_gamma, 6),
                'net_vega'  : round(net_vega, 4),
                'net_theta' : round(net_theta, 4),
                'iv'        : round(iv_now, 4),
                'trade_id'  : self._tid,
            })

            # ── Greek alerts ─────────────────────────────────────────────
            if abs(net_delta) > CFG['DELTA_ALERT']:
                self._add_alert(date_str, '⚡ DELTA BREACH',
                                round(net_delta, 4), CFG['DELTA_ALERT'],
                                'Hedge with NIFTY futures')

            if iv_chg_pct > CFG['VEGA_IV_ALERT_PCT']:
                self._add_alert(date_str, '🌊 VEGA / IV SPIKE',
                                round(iv_chg_pct * 100, 2),
                                CFG['VEGA_IV_ALERT_PCT'] * 100,
                                'Consider reducing position size')

            if abs(net_gamma) > CFG['GAMMA_ALERT']:
                self._add_alert(date_str, '⚠️  GAMMA SPIKE',
                                round(net_gamma, 6), CFG['GAMMA_ALERT'],
                                'Near-expiry risk — review position')

            # ── Exit conditions (same rules as PS) ───────────────────────
            profit_target = net_prem0 * lots * CFG['LOT_SIZE'] * CFG['IC_PROFIT_TARGET']
            sl_debit      = net_prem0 * CFG['IC_STOP_LOSS_MULT']

            reason = None
            if   day_i == 0                               : reason = None
            elif pnl >= profit_target                     : reason = 'PROFIT_TARGET'
            elif curr_net_prem >= sl_debit                : reason = 'STOP_LOSS'
            elif abs(net_delta) > CFG['DELTA_HARD_EXIT']  : reason = 'DELTA_EXIT'
            elif iv_chg_pct     > CFG['VEGA_IV_EXIT_PCT'] : reason = 'VEGA_EXIT'
            elif abs(net_gamma) > CFG['GAMMA_EXIT']       : reason = 'GAMMA_EXIT'
            elif dte_rem        <= CFG['IC_MIN_DTE']       : reason = 'TIME_EXIT'
            elif date           == expiry                  : reason = 'EXPIRY'

            if reason:
                self.cash += pnl
                trade.update({
                    'exit_date'        : date_str,
                    'exit_spot'        : round(S, 2),
                    'exit_net_premium' : round(curr_net_prem, 2),
                    'exit_reason'      : reason,
                    'exit_delta'       : round(net_delta, 4),
                    'exit_iv'          : round(iv_now, 4),
                    'pnl'              : round(pnl, 2),
                    'dte_held'         : day_i,
                })
                icon = '✅' if pnl >= 0 else '❌'
                print(f"     {icon} Exit [{reason:<16}]  "
                      f"P&L ₹{pnl:>+8,.0f}  |  DTE held: {day_i}")
                exited = True
                break

        if not exited:
            trade['exit_reason'] = 'OPEN_AT_END'
            trade['pnl']         = 0

        self.trades.append(trade)

    # ── Helpers ───────────────────────────────────────────────────────────
    def _add_alert(self, date, atype, value, threshold, action):
        self.alerts.append({
            'trade_id' : self._tid,
            'date'     : date,
            'type'     : atype,
            'value'    : value,
            'threshold': threshold,
            'action'   : action,
        })

    def _flush_pv_log(self):
        if not self.pv_log:
            return
        pv_dict  = {p['date']: p for p in self.pv_log}
        filled   = []
        last_val = CFG['INITIAL_CAPITAL']
        for d in self.df.index:
            ds = str(d.date())
            if ds in pv_dict:
                last_val = pv_dict[ds]['value']
                filled.append(pv_dict[ds])
            else:
                filled.append({'date': ds, 'value': last_val,
                               'open_pnl': 0, 'net_delta': 0,
                               'net_gamma': 0, 'net_vega': 0,
                               'net_theta': 0, 'iv': 0, 'trade_id': 0})
        self.pv_log = filled


# ═══════════════════════════════════════════════════════════════════════════
# SECTION 4B — LONG STRADDLE BACKTESTER
# ═══════════════════════════════════════════════════════════════════════════

class LongStraddleBacktester:
    """
    Long Straddle backtester — BUY ATM Call + BUY ATM Put.

    Opposite of short straddle:
      • Pay premium upfront (debit strategy)
      • Profit from BIG moves in either direction
      • Theta works AGAINST you (time decay hurts)
      • Vega works FOR you  (IV expansion helps)
      • Unlimited profit potential, max loss = premium paid

    Best deployed when:
      • IV is LOW relative to HV (cheap options)
      • Big move expected (earnings, budget, election)
      • Market is coiling / low volatility regime

    Greek profile (long position):
      Delta  ≈ 0      (delta neutral at ATM entry)
      Gamma  POSITIVE (profits accelerate on big moves)
      Vega   POSITIVE (benefits from IV expansion)
      Theta  NEGATIVE (bleeds premium every day)

    Exit rules (adapted from IIT-K PS framework):
      • Profit target : spot moves enough to return 40% on premium paid
      • Stop loss     : lose 50% of premium paid (time decay / no move)
      • Vega exit     : exit if IV drops >20% (IV crush kills long straddle)
      • Time exit     : exit at MIN_DTE (theta decay accelerates near expiry)
      • Entry filter  : IV must be LOW — enter only if IV < HV × 1.10
    """

    def __init__(self, df: pd.DataFrame):
        self.df     = df
        self.r      = CFG['RISK_FREE_RATE']
        self.cash   = CFG['INITIAL_CAPITAL']
        self.trades = []
        self.alerts = []
        self.pv_log = []
        self._tid   = 0

    # ── PUBLIC ────────────────────────────────────────────────────────────
    def run(self, from_date: str, to_date: str):
        expiries = get_monthly_expiry_dates(self.df.index, from_date, to_date)
        print(f"\n📅  {len(expiries)} monthly expiry cycles to simulate\n")
        for exp in expiries:
            self._run_cycle(exp)
        self._flush_pv_log()
        print(f"\n✅  Backtest complete — {len(self.trades)} trades")
        return self.trades, self.alerts, self.pv_log

    # ── PRIVATE — one expiry cycle ────────────────────────────────────────
    def _run_cycle(self, expiry: pd.Timestamp):
        all_dates  = self.df.index
        pre_expiry = all_dates[all_dates < expiry]

        if len(pre_expiry) < CFG['ENTRY_DTE'] + 5:
            return

        entry_date = pre_expiry[-CFG['ENTRY_DTE']]
        entry_row  = self.df.loc[entry_date]
        S0         = float(entry_row['Close'])
        iv0        = float(entry_row['IV'])
        hv0        = float(entry_row['HV20'])

        # Long straddle: enter only when IV is CHEAP relative to HV
        # IV/HV < 0.95 means options are underpriced — ideal for buying
        iv_hv_ratio = float(entry_row.get('IV_HV_ratio', iv0 / hv0))
        if iv_hv_ratio > CFG['LS_IV_HV_MAX']:
            print(f"  ⏭   {expiry.date()} — IV/HV={iv_hv_ratio:.2f} > {CFG['LS_IV_HV_MAX']} (vol too expensive for long), skip")
            return

        K  = round(S0 / CFG['STRIKE_ROUND']) * CFG['STRIKE_ROUND']
        T0 = CFG['ENTRY_DTE'] / 365

        # BUY ATM Call + BUY ATM Put
        ce0 = greeks(S0, K, T0, self.r, iv0, 'CE')
        pe0 = greeks(S0, K, T0, self.r, iv0, 'PE')

        prem0   = ce0['price'] + pe0['price']   # total debit paid
        net_d0  = ce0['delta'] + pe0['delta']   # ~0 for ATM
        net_g0  = ce0['gamma'] + pe0['gamma']   # positive (good)
        net_v0  = ce0['vega']  + pe0['vega']    # positive (good)
        net_t0  = ce0['theta'] + pe0['theta']   # negative (bleeds)

        if prem0 <= 0:
            return

        # Lot sizing: risk only what we're willing to lose (premium paid = max loss)
        risk   = self.cash * CFG['CAPITAL_RISK_PCT']
        lots   = max(1, int(risk / (prem0 * CFG['LOT_SIZE'])))
        lots   = min(lots, CFG['MAX_LOTS'])

        self._tid += 1
        trade = {
            'id'              : self._tid,
            'strategy'        : 'Long Straddle',
            'expiry'          : str(expiry.date()),
            'entry_date'      : str(entry_date.date()),
            'entry_spot'      : round(S0, 2),
            'strike'          : K,
            'entry_iv'        : round(iv0, 4),
            'entry_hv'        : round(hv0, 4),
            'entry_ce_px'     : round(ce0['price'], 2),
            'entry_pe_px'     : round(pe0['price'], 2),
            'entry_premium'   : round(prem0, 2),
            'entry_delta'     : round(net_d0, 4),
            'entry_gamma'     : round(net_g0, 5),
            'entry_vega'      : round(net_v0, 4),
            'entry_theta'     : round(net_t0, 4),
            'lots'            : lots,
            'exit_date'       : None,
            'exit_spot'       : None,
            'exit_premium'    : None,
            'exit_reason'     : None,
            'exit_delta'      : None,
            'exit_iv'         : None,
            'pnl'             : None,
            'dte_held'        : None,
        }

        print(f"  ▶  Entry {entry_date.date()} | Expiry {expiry.date()} | "
              f"Spot ₹{S0:.0f} | Strike ₹{K} | "
              f"Debit ₹{prem0:.1f} | Lots {lots}")

        # ── Daily monitoring ─────────────────────────────────────────────
        cycle_dates = all_dates[(all_dates >= entry_date) & (all_dates <= expiry)]
        exited      = False

        for day_i, date in enumerate(cycle_dates):
            row     = self.df.loc[date]
            S       = float(row['Close'])
            iv_now  = float(row['IV'])
            dte_rem = len(cycle_dates) - day_i

            T = max(dte_rem / 365, 1e-6)

            ce_now = greeks(S, K, T, self.r, iv_now, 'CE')
            pe_now = greeks(S, K, T, self.r, iv_now, 'PE')

            curr_prem = ce_now['price'] + pe_now['price']

            # Long straddle P&L = current value - premium paid
            pnl = (curr_prem - prem0) * lots * CFG['LOT_SIZE']

            # Net Greeks (long position — no negation)
            net_delta = ce_now['delta'] + pe_now['delta']
            net_gamma = ce_now['gamma'] + pe_now['gamma']
            net_vega  = ce_now['vega']  + pe_now['vega']
            net_theta = ce_now['theta'] + pe_now['theta']

            iv_chg_pct = (iv_now - iv0) / iv0
            date_str   = str(date.date())

            # ── Portfolio value log ──────────────────────────────────────
            self.pv_log.append({
                'date'      : date_str,
                'value'     : round(self.cash + pnl, 2),
                'open_pnl'  : round(pnl, 2),
                'net_delta' : round(net_delta, 4),
                'net_gamma' : round(net_gamma, 6),
                'net_vega'  : round(net_vega, 4),
                'net_theta' : round(net_theta, 4),
                'iv'        : round(iv_now, 4),
                'trade_id'  : self._tid,
            })

            # ── Greek alerts (inverted logic for long position) ──────────
            # For long straddle: warn on THETA bleed and IV DROP (not spike)
            daily_theta_bleed = net_theta * CFG['LOT_SIZE'] * lots
            if daily_theta_bleed < -500:   # losing >₹500/day to theta
                self._add_alert(date_str, '🕐 THETA BLEED',
                                round(daily_theta_bleed, 0), -500,
                                'Heavy theta decay — consider exiting')

            # ── Greek alerts ─────────────────────────────────────────────
            daily_theta_bleed = net_theta * CFG['LOT_SIZE'] * lots
            if daily_theta_bleed < -500:
                self._add_alert(date_str, '🕐 THETA BLEED',
                                round(daily_theta_bleed, 0), -500,
                                'Heavy theta decay — consider exiting')

            if iv_chg_pct < -0.12:
                self._add_alert(date_str, '📉 IV CRUSH',
                                round(iv_chg_pct * 100, 2), -12,
                                'IV falling — hurts long vega position')

            if abs(net_delta) > 0.40:
                self._add_alert(date_str, '⚡ DELTA DRIFT',
                                round(net_delta, 4), 0.40,
                                'Position going directional — lock in winner')

            # ── Exit conditions ──────────────────────────────────────────
            profit_target  =  prem0 * lots * CFG['LOT_SIZE'] * CFG['LS_PROFIT_TARGET']
            max_loss_limit = -prem0 * lots * CFG['LOT_SIZE'] * CFG['LS_STOP_LOSS_PCT']
            iv_crush_exit  = iv_chg_pct < -CFG['LS_IV_CRUSH_EXIT']
            too_long       = day_i >= CFG['LS_MAX_HOLD_DAYS']

            # Leg-lock: if one side is deep ITM (|delta| > 0.60),
            # the straddle is acting like a naked option — take profit now
            call_delta = ce_now['delta']
            put_delta  = pe_now['delta']   # negative
            leg_lock   = (call_delta > 0.65 or put_delta < -0.65)

            reason = None
            if   day_i == 0               : reason = None
            elif pnl >= profit_target      : reason = 'PROFIT_TARGET'
            elif leg_lock and pnl > 0      : reason = 'LEG_LOCK_PROFIT'
            elif pnl <= max_loss_limit     : reason = 'STOP_LOSS_25PCT'
            elif iv_crush_exit             : reason = 'IV_CRUSH_EXIT'
            elif too_long                  : reason = 'MAX_HOLD_EXIT'
            elif dte_rem <= CFG['LS_MIN_DTE']: reason = 'TIME_EXIT'
            elif date    == expiry          : reason = 'EXPIRY'

            if reason:
                self.cash += pnl
                trade.update({
                    'exit_date'    : date_str,
                    'exit_spot'    : round(S, 2),
                    'exit_ce_px'   : round(ce_now['price'], 2),
                    'exit_pe_px'   : round(pe_now['price'], 2),
                    'exit_premium' : round(curr_prem, 2),
                    'exit_reason'  : reason,
                    'exit_delta'   : round(net_delta, 4),
                    'exit_iv'      : round(iv_now, 4),
                    'pnl'          : round(pnl, 2),
                    'dte_held'     : day_i,
                })
                icon = '✅' if pnl >= 0 else '❌'
                print(f"     {icon} Exit [{reason:<18}]  "
                      f"P&L ₹{pnl:>+8,.0f}  |  DTE held: {day_i}")
                exited = True
                break

        if not exited:
            trade['exit_reason'] = 'OPEN_AT_END'
            trade['pnl']         = 0

        self.trades.append(trade)

    def _add_alert(self, date, atype, value, threshold, action):
        self.alerts.append({
            'trade_id' : self._tid,
            'date'     : date,
            'type'     : atype,
            'value'    : value,
            'threshold': threshold,
            'action'   : action,
        })

    def _flush_pv_log(self):
        if not self.pv_log:
            return
        pv_dict  = {p['date']: p for p in self.pv_log}
        filled   = []
        last_val = CFG['INITIAL_CAPITAL']
        for d in self.df.index:
            ds = str(d.date())
            if ds in pv_dict:
                last_val = pv_dict[ds]['value']
                filled.append(pv_dict[ds])
            else:
                filled.append({'date': ds, 'value': last_val,
                               'open_pnl': 0, 'net_delta': 0,
                               'net_gamma': 0, 'net_vega': 0,
                               'net_theta': 0, 'iv': 0, 'trade_id': 0})
        self.pv_log = filled


# ═══════════════════════════════════════════════════════════════════════════
# SECTION 5 — METRICS
# ═══════════════════════════════════════════════════════════════════════════

def compute_metrics(trades, pv_log, initial_capital):
    closed = [t for t in trades
              if t.get('pnl') is not None and t['exit_reason'] != 'OPEN_AT_END']
    if not closed:
        print("⚠️  No closed trades found — check date range or IV filter.")
        return {}

    pnls    = [t['pnl'] for t in closed]
    wins    = [p for p in pnls if p > 0]
    losses  = [p for p in pnls if p <= 0]

    pv      = pd.Series([p['value'] for p in pv_log])
    rets    = pv.pct_change().dropna()
    roll_mx = pv.cummax()
    dd      = ((pv - roll_mx) / roll_mx * 100).round(3)

    exit_counts = {}
    for t in closed:
        r = t.get('exit_reason', 'UNKNOWN')
        exit_counts[r] = exit_counts.get(r, 0) + 1

    sharpe = (rets.mean() / rets.std() * np.sqrt(252)
              if rets.std() > 0 else 0)

    return {
        'total_pnl'        : round(sum(pnls), 2),
        'total_return_pct' : round(sum(pnls) / initial_capital * 100, 2),
        'num_trades'       : len(closed),
        'win_rate'         : round(len(wins) / len(pnls) * 100, 1),
        'avg_win'          : round(np.mean(wins), 2)    if wins   else 0,
        'avg_loss'         : round(np.mean(losses), 2)  if losses else 0,
        'best_trade'       : round(max(pnls), 2),
        'worst_trade'      : round(min(pnls), 2),
        'profit_factor'    : round(sum(wins) / abs(sum(losses)), 2)
                             if losses and sum(losses) != 0 else 999,
        'sharpe'           : round(sharpe, 2),
        'max_drawdown_pct' : round(dd.min(), 2),
        'exit_counts'      : exit_counts,
        'drawdown_series'  : dd.tolist(),
    }


# ═══════════════════════════════════════════════════════════════════════════
# SECTION 6 — PLOTLY DASHBOARD
# ═══════════════════════════════════════════════════════════════════════════

def plot_dashboard(trades, alerts, pv_log, metrics, initial_capital):
    if not pv_log or not metrics:
        print("⚠️  Nothing to plot.")
        return

    dates   = [p['date']  for p in pv_log]
    values  = [p['value'] for p in pv_log]
    dd_s    = metrics.get('drawdown_series', [])
    deltas  = [p['net_delta'] for p in pv_log]
    ivs     = [p['iv'] * 100  for p in pv_log]

    closed  = [t for t in trades
               if t.get('pnl') is not None and t['exit_reason'] != 'OPEN_AT_END']
    pnls    = [t['pnl'] for t in closed]
    t_dates = [t['entry_date'] for t in closed]
    bar_colors = ['#00d4a0' if p >= 0 else '#ff4560' for p in pnls]

    alert_colors = {'⚡ DELTA BREACH'   : '#f5a623',
                    '🌊 VEGA / IV SPIKE' : '#e74c3c',
                    '⚠️  GAMMA SPIKE'    : '#9b59b6'}

    # ── Build figure with explicit specs — pie at row2,col2 ────────────
    fig = make_subplots(
        rows=3, cols=2,
        subplot_titles=(
            '📈 Portfolio Value (₹)',   '📉 Drawdown from Peak (%)',
            '📊 Per-Trade P&L (₹)',     '🥧 Exit Reason Breakdown',
            '⚡ Net Portfolio Delta',   '🌡️  Implied Volatility (%)',
        ),
        specs=[
            [{'type': 'xy'},  {'type': 'xy'}],
            [{'type': 'xy'},  {'type': 'pie'}],   # pie must be declared here
            [{'type': 'xy'},  {'type': 'xy'}],
        ],
        vertical_spacing=0.13,
        horizontal_spacing=0.09,
    )

    # ── 1. Portfolio value ─────────────────────────────────────────────
    fig.add_trace(go.Scatter(
        x=dates, y=values, mode='lines', name='Portfolio',
        line=dict(color='#00d4a0', width=2),
        fill='tozeroy', fillcolor='rgba(0,212,160,0.07)',
    ), row=1, col=1)
    fig.add_shape(type='line', xref='paper', yref='y1',
                  x0=0, x1=1, y0=initial_capital, y1=initial_capital,
                  line=dict(dash='dash', color='rgba(255,255,255,0.3)'))

    # ── 2. Drawdown ────────────────────────────────────────────────────
    if dd_s and len(dd_s) == len(dates):
        fig.add_trace(go.Scatter(
            x=dates, y=dd_s, mode='lines', name='Drawdown',
            line=dict(color='#ff4560', width=1.5),
            fill='tozeroy', fillcolor='rgba(255,69,96,0.09)',
        ), row=1, col=2)

    # ── 3. Trade P&L bars ──────────────────────────────────────────────
    fig.add_trace(go.Bar(
        x=t_dates, y=pnls, name='Trade P&L',
        marker_color=bar_colors,
        text=[f'₹{p:+,.0f}' for p in pnls],
        textposition='outside',
        textfont=dict(size=9),
    ), row=2, col=1)

    # ── 4. Exit reason pie ─────────────────────────────────────────────
    ec = metrics.get('exit_counts', {})
    if ec:
        fig.add_trace(go.Pie(
            labels=list(ec.keys()),
            values=list(ec.values()),
            hole=0.42,
            textinfo='label+percent',
            textfont=dict(size=9),
            marker=dict(colors=[
                '#00d4a0','#ff4560','#f5a623',
                '#4a90e2','#9b59b6','#e67e22','#1abc9c'
            ]),
        ), row=2, col=2)

    # ── 5. Net Delta ───────────────────────────────────────────────────
    fig.add_trace(go.Scatter(
        x=dates, y=deltas, mode='lines', name='Net Δ',
        line=dict(color='#4a90e2', width=1.4),
    ), row=3, col=1)
    # Reference lines for delta thresholds (using shapes instead of add_hline)
    for y_val, label in [(CFG['DELTA_ALERT'],  '+0.20 hedge'),
                         (-CFG['DELTA_ALERT'], '-0.20 hedge'),
                         (0,                   '')]:
        fig.add_shape(
            type='line', xref='paper',
            yref='y5',                 # y5 = row3,col1 in a 3×2 grid
            x0=0, x1=1, y0=y_val, y1=y_val,
            line=dict(
                dash='dot' if y_val != 0 else 'solid',
                color='rgba(255,69,96,0.6)' if y_val != 0
                      else 'rgba(255,255,255,0.15)',
                width=1,
            )
        )

    # ── 6. IV ──────────────────────────────────────────────────────────
    fig.add_trace(go.Scatter(
        x=dates, y=ivs, mode='lines', name='IV (%)',
        line=dict(color='#f5a623', width=1.4),
        fill='tozeroy', fillcolor='rgba(245,166,35,0.06)',
    ), row=3, col=2)

    # Alert markers on IV chart
    pv_date_map = {p['date']: p['iv'] * 100 for p in pv_log}
    for atype, acolor in alert_colors.items():
        a_dates = [a['date'] for a in alerts if a['type'] == atype]
        a_ivs   = [pv_date_map.get(d, 0) for d in a_dates]
        if a_dates:
            fig.add_trace(go.Scatter(
                x=a_dates, y=a_ivs, mode='markers', name=atype,
                marker=dict(color=acolor, size=8, symbol='x'),
            ), row=3, col=2)

    # ── Layout ─────────────────────────────────────────────────────────
    stats_text = (
        f"Trades: {metrics['num_trades']}  |  "
        f"Win: {metrics['win_rate']}%  |  "
        f"P&L: ₹{metrics['total_pnl']:+,.0f}  |  "
        f"Return: {metrics['total_return_pct']:+.1f}%  |  "
        f"Sharpe: {metrics['sharpe']}  |  "
        f"MaxDD: {metrics['max_drawdown_pct']}%  |  "
        f"PF: {metrics['profit_factor']}"
    )

    fig.update_layout(
        title=dict(
            text=f'<b>NIFTY Iron Condor — Greek-Based Backtester</b>'
                 f'<br><sup style="color:#94a3b8">{stats_text}</sup>',
            font=dict(size=15, color='#e2e8f0'),
            x=0.5, xanchor='center',
        ),
        paper_bgcolor='#0f172a',
        plot_bgcolor ='#1e293b',
        font=dict(color='#cbd5e1',
                  family='JetBrains Mono, Courier New, monospace'),
        height=950,
        showlegend=False,
        margin=dict(t=110, b=40, l=55, r=30),
    )

    # Style subplot title annotations
    for ann in fig['layout']['annotations']:
        ann['font'] = dict(size=11, color='#94a3b8')

    # Apply axis styling ONLY to xy subplots — skip the pie (row2, col2)
    axis_style = dict(gridcolor='#334155', zeroline=False,
                      tickfont=dict(size=9), showgrid=True)
    for r, c in [(1,1), (1,2), (2,1), (3,1), (3,2)]:
        fig.update_xaxes(axis_style, row=r, col=c)
        fig.update_yaxes(axis_style, row=r, col=c)

    fig.show()

    out_path = 'greek_backtest_dashboard.html'
    fig.write_html(out_path)
    print(f"\n💾  Dashboard saved → {out_path}")


# ═══════════════════════════════════════════════════════════════════════════
# SECTION 7 — PRINT REPORTS
# ═══════════════════════════════════════════════════════════════════════════

def print_summary(metrics, initial_capital):
    sep = '═' * 54
    print(f"\n{sep}")
    print("  BACKTEST SUMMARY — Greek Short Straddle (NIFTY)")
    print(sep)
    rows = [
        ('Initial Capital',  f"₹{initial_capital:>12,.0f}"),
        ('Total P&L',        f"₹{metrics['total_pnl']:>+12,.0f}"),
        ('Total Return',     f"{metrics['total_return_pct']:>+11.2f}%"),
        ('No. of Trades',    f"{metrics['num_trades']:>13}"),
        ('Win Rate',         f"{metrics['win_rate']:>12.1f}%"),
        ('Avg Win',          f"₹{metrics['avg_win']:>+12,.0f}"),
        ('Avg Loss',         f"₹{metrics['avg_loss']:>+12,.0f}"),
        ('Best Trade',       f"₹{metrics['best_trade']:>+12,.0f}"),
        ('Worst Trade',      f"₹{metrics['worst_trade']:>+12,.0f}"),
        ('Profit Factor',    f"{metrics['profit_factor']:>13.2f}"),
        ('Sharpe Ratio',     f"{metrics['sharpe']:>13.2f}"),
        ('Max Drawdown',     f"{metrics['max_drawdown_pct']:>12.2f}%"),
    ]
    for label, val in rows:
        print(f"  {label:<20}: {val}")
    print('─' * 54)
    print("  Exit Breakdown:")
    for r, c in metrics.get('exit_counts', {}).items():
        print(f"    {r:<28}: {c}")
    print(sep)


def print_trade_log(trades):
    closed = [t for t in trades if t.get('exit_reason') != 'OPEN_AT_END']
    w = 120
    print(f"\n{'─'*w}")
    print(f"  {'#':>3}  {'Entry':>12}  {'Exit':>12}  "
          f"{'Condor Range':>20}  "
          f"{'Lots':>4}  {'NetPrem':>8}  {'P&L (₹)':>10}  "
          f"{'Exit Δ':>8}  {'DTE':>5}  Exit Reason")
    print(f"{'─'*w}")
    for t in closed:
        condor_range = (f"{t.get('K_short_put','?')}/"
                        f"{t.get('K_short_call','?')}")
        print(
            f"  {t['id']:>3}  "
            f"{t['entry_date']:>12}  "
            f"{str(t.get('exit_date','—')):>12}  "
            f"{condor_range:>20}  "
            f"{t['lots']:>4}  "
            f"{t['entry_net_premium']:>8.1f}  "
            f"{t.get('pnl', 0):>+10,.0f}  "
            f"{t.get('exit_delta', 0):>+8.4f}  "
            f"{t.get('dte_held', 0):>5}  "
            f"{t.get('exit_reason','?')}"
        )
    print(f"{'─'*w}")


def print_alert_log(alerts):
    if not alerts:
        print("\n  ✅  No Greek threshold breaches during backtest.")
        return
    w = 92
    print(f"\n{'─'*w}")
    print(f"  {'TID':>4}  {'Date':>12}  {'Alert Type':<24}  "
          f"{'Value':>9}  {'Threshold':>9}  Action")
    print(f"{'─'*w}")
    for a in alerts:
        print(
            f"  {a['trade_id']:>4}  "
            f"{a['date']:>12}  "
            f"{a['type']:<24}  "
            f"{a['value']:>9}  "
            f"{a['threshold']:>9}  "
            f"{a['action']}"
        )
    print(f"{'─'*w}")
    print(f"  Total alerts: {len(alerts)}")


# ═══════════════════════════════════════════════════════════════════════════
# SECTION 8 — MAIN ENTRY POINT
# ═══════════════════════════════════════════════════════════════════════════

def plot_comparison_dashboard(results: dict, initial_capital: float):
    """
    Side-by-side comparison dashboard for Iron Condor vs Long Straddle.
    results = { 'Iron Condor': (trades, alerts, pv_log, metrics),
                'Long Straddle': (trades, alerts, pv_log, metrics) }
    """
    colors = {'Iron Condor': '#00d4a0', 'Long Straddle': '#f5a623'}

    fig = make_subplots(
        rows=3, cols=2,
        subplot_titles=(
            '📈 Portfolio Value — Iron Condor',
            '📈 Portfolio Value — Long Straddle',
            '📉 Drawdown — Iron Condor',
            '📉 Drawdown — Long Straddle',
            '📊 Per-Trade P&L — Iron Condor',
            '📊 Per-Trade P&L — Long Straddle',
        ),
        specs=[
            [{'type': 'xy'}, {'type': 'xy'}],
            [{'type': 'xy'}, {'type': 'xy'}],
            [{'type': 'xy'}, {'type': 'xy'}],
        ],
        vertical_spacing=0.12,
        horizontal_spacing=0.08,
    )

    col_map = {'Iron Condor': 1, 'Long Straddle': 2}

    for strat, (trades, alerts, pv_log, metrics) in results.items():
        if not pv_log or not metrics:
            continue
        col   = col_map[strat]
        color = colors[strat]
        dates = [p['date']  for p in pv_log]
        vals  = [p['value'] for p in pv_log]
        dd_s  = metrics.get('drawdown_series', [])

        closed  = [t for t in trades if t.get('pnl') is not None
                   and t.get('exit_reason') != 'OPEN_AT_END']
        pnls    = [t['pnl'] for t in closed]
        t_dates = [t['entry_date'] for t in closed]
        bar_col = ['#00d4a0' if p >= 0 else '#ff4560' for p in pnls]

        # Row 1: portfolio value
        fig.add_trace(go.Scatter(
            x=dates, y=vals, mode='lines', name=f'{strat} PV',
            line=dict(color=color, width=2),
            fill='tozeroy', fillcolor=f'rgba({",".join(str(int(c*255)) for c in __import__("matplotlib.colors", fromlist=["to_rgb"]).to_rgb(color))},0.08)' if False else 'rgba(0,212,160,0.07)' if strat == 'Iron Condor' else 'rgba(245,166,35,0.07)',
        ), row=1, col=col)
        fig.add_shape(type='line', xref='paper',
                      yref=f'y{1 if col==1 else 2}',
                      x0=0, x1=1, y0=initial_capital, y1=initial_capital,
                      line=dict(dash='dash', color='rgba(255,255,255,0.25)', width=1))

        # Row 2: drawdown
        if dd_s and len(dd_s) == len(dates):
            fig.add_trace(go.Scatter(
                x=dates, y=dd_s, mode='lines',
                name=f'{strat} DD',
                line=dict(color='#ff4560', width=1.4),
                fill='tozeroy', fillcolor='rgba(255,69,96,0.09)',
            ), row=2, col=col)

        # Row 3: per-trade bars
        if pnls:
            fig.add_trace(go.Bar(
                x=t_dates, y=pnls, name=f'{strat} Trades',
                marker_color=bar_col,
                text=[f'₹{p:+,.0f}' for p in pnls],
                textposition='outside',
                textfont=dict(size=8),
            ), row=3, col=col)

    # Stats subtitle
    def stat_line(m, name):
        if not m:
            return f'{name}: no data'
        return (f"<b>{name}</b>  Trades:{m['num_trades']}  "
                f"Win:{m['win_rate']}%  "
                f"P&L:₹{m['total_pnl']:+,.0f}  "
                f"Ret:{m['total_return_pct']:+.1f}%  "
                f"Sharpe:{m['sharpe']}  "
                f"MaxDD:{m['max_drawdown_pct']}%")

    ic_m  = results['Iron Condor'][3]
    ls_m  = results['Long Straddle'][3]
    sub   = stat_line(ic_m, '🟢 Iron Condor') + '<br>' + stat_line(ls_m, '🟡 Long Straddle')

    fig.update_layout(
        title=dict(
            text='<b>NIFTY Strategy Comparison — Iron Condor vs Long Straddle</b>'
                 f'<br><sup style="color:#94a3b8">{sub}</sup>',
            font=dict(size=14, color='#e2e8f0'),
            x=0.5, xanchor='center',
        ),
        paper_bgcolor='#0f172a',
        plot_bgcolor ='#1e293b',
        font=dict(color='#cbd5e1',
                  family='JetBrains Mono, Courier New, monospace'),
        height=1000,
        showlegend=False,
        margin=dict(t=130, b=40, l=55, r=30),
    )
    for ann in fig['layout']['annotations']:
        ann['font'] = dict(size=10, color='#94a3b8')

    axis_style = dict(gridcolor='#334155', zeroline=False,
                      tickfont=dict(size=8), showgrid=True)
    for r in range(1, 4):
        for c in range(1, 3):
            fig.update_xaxes(axis_style, row=r, col=c)
            fig.update_yaxes(axis_style, row=r, col=c)

    fig.show()
    out = 'comparison_dashboard.html'
    fig.write_html(out)
    print(f"\n💾  Comparison dashboard saved → {out}")


def print_comparison_summary(results: dict, initial_capital: float):
    sep = '═' * 62
    print(f"\n{sep}")
    print("  STRATEGY COMPARISON SUMMARY")
    print(sep)
    labels = ['Iron Condor', 'Long Straddle']
    metrics_list = [results[l][3] for l in labels]

    rows = [
        ('Metric',          'Iron Condor',    'Long Straddle'),
        ('─'*20,            '─'*15,           '─'*15),
        ('No. of Trades',   None, None),
        ('Win Rate',        None, None),
        ('Total P&L (₹)',   None, None),
        ('Total Return %',  None, None),
        ('Avg Win (₹)',     None, None),
        ('Avg Loss (₹)',    None, None),
        ('Profit Factor',   None, None),
        ('Sharpe Ratio',    None, None),
        ('Max Drawdown %',  None, None),
    ]

    keys = [None, None,
            'num_trades', 'win_rate', 'total_pnl',
            'total_return_pct', 'avg_win', 'avg_loss',
            'profit_factor', 'sharpe', 'max_drawdown_pct']

    print(f"  {'Metric':<22} {'Iron Condor':>16} {'Long Straddle':>16}")
    print(f"  {'─'*22} {'─'*16} {'─'*16}")
    for label, key in zip(
        ['No. of Trades','Win Rate (%)','Total P&L (₹)',
         'Total Return (%)','Avg Win (₹)','Avg Loss (₹)',
         'Profit Factor','Sharpe Ratio','Max Drawdown (%)'],
        ['num_trades','win_rate','total_pnl',
         'total_return_pct','avg_win','avg_loss',
         'profit_factor','sharpe','max_drawdown_pct']
    ):
        vals = []
        for m in metrics_list:
            v = m.get(key, 'N/A') if m else 'N/A'
            vals.append(f'{v:>16}' if isinstance(v, str) else f'{v:>16,.2f}')
        print(f"  {label:<22} {vals[0]} {vals[1]}")
    print(sep)


def run_backtest(from_date=None, to_date=None):
    """Run Iron Condor only."""
    from_date = from_date or CFG['FROM_DATE']
    to_date   = to_date   or CFG['TO_DATE']

    print("╔══════════════════════════════════════════════════════╗")
    print("║  NIFTY Iron Condor — Greek-Based Backtester          ║")
    print("║  Data: yfinance  |  Greeks: Black-Scholes            ║")
    print("╚══════════════════════════════════════════════════════╝")

    df = fetch_nifty(from_date, to_date)
    bt = IronCondorBacktester(df)
    trades, alerts, pv_log = bt.run(from_date, to_date)
    metrics = compute_metrics(trades, pv_log, CFG['INITIAL_CAPITAL'])
    if not metrics:
        return trades, alerts, pv_log, {}

    print_summary(metrics, CFG['INITIAL_CAPITAL'])
    print("\n📋  TRADE LOG")
    print_trade_log(trades)
    print("\n⚡  GREEK ALERTS LOG")
    print_alert_log(alerts)
    plot_dashboard(trades, alerts, pv_log, metrics, CFG['INITIAL_CAPITAL'])

    with open('backtest_results_ic.json', 'w') as f:
        json.dump(dict(metrics=metrics, trades=trades,
                       alerts=alerts, pv_log=pv_log), f, indent=2, default=str)
    print("\n💾  Results saved → backtest_results_ic.json")
    return trades, alerts, pv_log, metrics


def run_long_straddle(from_date=None, to_date=None):
    """Run Long Straddle only."""
    from_date = from_date or CFG['FROM_DATE']
    to_date   = to_date   or CFG['TO_DATE']

    print("╔══════════════════════════════════════════════════════╗")
    print("║  NIFTY Long Straddle — Greek-Based Backtester        ║")
    print("║  Data: yfinance  |  Greeks: Black-Scholes            ║")
    print("╚══════════════════════════════════════════════════════╝")

    df = fetch_nifty(from_date, to_date)
    bt = LongStraddleBacktester(df)
    trades, alerts, pv_log = bt.run(from_date, to_date)
    metrics = compute_metrics(trades, pv_log, CFG['INITIAL_CAPITAL'])
    if not metrics:
        return trades, alerts, pv_log, {}

    print_summary(metrics, CFG['INITIAL_CAPITAL'])
    print("\n📋  TRADE LOG")
    print_trade_log(trades)
    print("\n⚡  GREEK ALERTS LOG")
    print_alert_log(alerts)
    plot_dashboard(trades, alerts, pv_log, metrics, CFG['INITIAL_CAPITAL'])

    with open('backtest_results_ls.json', 'w') as f:
        json.dump(dict(metrics=metrics, trades=trades,
                       alerts=alerts, pv_log=pv_log), f, indent=2, default=str)
    print("\n💾  Results saved → backtest_results_ls.json")
    return trades, alerts, pv_log, metrics


def run_comparison(from_date=None, to_date=None):
    """
    Run BOTH strategies and show a side-by-side comparison.
    This is the recommended entry point.
    """
    from_date = from_date or CFG['FROM_DATE']
    to_date   = to_date   or CFG['TO_DATE']

    print("╔══════════════════════════════════════════════════════╗")
    print("║  NIFTY Strategy Comparison                           ║")
    print("║  Iron Condor  vs  Long Straddle                      ║")
    print("║  Data: yfinance  |  Greeks: Black-Scholes            ║")
    print("╚══════════════════════════════════════════════════════╝")

    df = fetch_nifty(from_date, to_date)

    # ── Iron Condor ───────────────────────────────────────────────────
    print("\n" + "─"*54)
    print("  Running: Iron Condor")
    print("─"*54)
    ic_bt = IronCondorBacktester(df)
    ic_trades, ic_alerts, ic_pv = ic_bt.run(from_date, to_date)
    ic_metrics = compute_metrics(ic_trades, ic_pv, CFG['INITIAL_CAPITAL'])

    # ── Long Straddle ─────────────────────────────────────────────────
    print("\n" + "─"*54)
    print("  Running: Long Straddle")
    print("─"*54)
    ls_bt = LongStraddleBacktester(df)
    ls_trades, ls_alerts, ls_pv = ls_bt.run(from_date, to_date)
    ls_metrics = compute_metrics(ls_trades, ls_pv, CFG['INITIAL_CAPITAL'])

    results = {
        'Iron Condor'   : (ic_trades, ic_alerts, ic_pv, ic_metrics),
        'Long Straddle' : (ls_trades, ls_alerts, ls_pv, ls_metrics),
    }

    # ── Print comparison ──────────────────────────────────────────────
    print_comparison_summary(results, CFG['INITIAL_CAPITAL'])

    # ── Comparison dashboard ──────────────────────────────────────────
    plot_comparison_dashboard(results, CFG['INITIAL_CAPITAL'])

    # ── Save JSON ─────────────────────────────────────────────────────
    with open('comparison_results.json', 'w') as f:
        json.dump({
            'iron_condor'   : dict(metrics=ic_metrics, trades=ic_trades),
            'long_straddle' : dict(metrics=ls_metrics, trades=ls_trades),
        }, f, indent=2, default=str)
    print("\n💾  Comparison results saved → comparison_results.json")

    return results


# ── Auto-run ──────────────────────────────────────────────────────────────
if __name__ == '__main__':
    # Run both strategies and compare
    # To run individually:
    #   run_backtest()          ← Iron Condor only
    #   run_long_straddle()     ← Long Straddle only
    #   run_comparison()        ← Both + side-by-side dashboard
    results = run_comparison()
